# Distributed Text Processing with TaskVine

A starter notebook for running distributed tasks with [TaskVine](https://cctools.readthedocs.io/en/latest/taskvine/) on Floability.

**What this template does:**
1. Connects to a TaskVine manager
2. Uses one local backpack file and one remote Project Gutenberg file staged by Floability
3. Declares both inputs and processes them in parallel across workers
4. Collects and summarizes results

**To customize:** edit `worker_function` with your own logic.


## Setup: Manager Connection

TaskVine requires a manager process. The manager name and ports come from environment variables set by Floability.

In [ ]:
import os
import ndcctools.taskvine as vine

def parse_ports(ports_str: str) -> list[int]:
    ports = [int(p.strip()) for p in ports_str.split(",") if p.strip()]

    if not ports:
        raise ValueError("No valid ports provided")

    return ports

# Get manager info from environment (set by Floability)
manager_name = os.environ.get('VINE_MANAGER_NAME')
manager_ports = parse_ports(os.environ.get("VINE_MANAGER_PORTS", "9123,9150"))

print(f'Manager Name: {manager_name}')
print(f'Manager Ports: {manager_ports}')

m = vine.Manager(manager_ports, name=manager_name)
print(f"[manager] Listening on port {m.port}")

## Define Worker Function

The worker function runs on distributed worker nodes. It calculates line and word totals, then counts occurrences of `war` and `peace`.

**To customize:** Replace the function body with your processing logic.

In [ ]:
def worker_function(file_path, keywords=("war", "peace")):
    """Return basic text statistics and keyword counts for one file."""
    import re
    from pathlib import Path

    path = Path(file_path)
    if not path.exists():
        return {'file': file_path, 'error': 'File not found'}

    text = path.read_text(encoding="utf-8", errors="replace")
    words = re.findall(r"\b[\w']+\b", text.casefold())
    return {
        'file': path.name,
        'line_count': len(text.splitlines()),
        'word_count': len(words),
        'keyword_counts': {keyword: words.count(keyword.casefold()) for keyword in keywords},
    }

# Test the function locally
print('Worker function defined.')

## Submit Tasks to Workers

Floability has already placed the bundled local file and downloaded remote file at predictable paths. This cell declares those staged files and submits one task per file.

**To customize:** Modify the file list to match your `data/` directory contents.

In [ ]:
import glob

DATA_DIR = "data/text_data"

files = sorted(glob.glob(os.path.join(DATA_DIR, "*.txt"))) if os.path.isdir(DATA_DIR) else []
print(f"Found {len(files)} file(s) in {DATA_DIR}/")

if not files:
    raise RuntimeError("No staged text files found. Check data/data.yml and Floability's data logs.")

declared = {path: m.declare_file(path) for path in files}

task_map = {}
for file_path in files:
    t = vine.PythonTask(worker_function, file_path)
    t.add_input(declared[file_path], file_path)
    tid = m.submit(t)
    task_map[tid] = file_path

print(f"Submitted {len(task_map)} task(s) to the manager")

## Collect Results

Wait for tasks to complete and collect results.

In [ ]:
results = []

while not m.empty():
    done = m.wait(5)
    if not done:
        continue
    if done.successful():
        r = done.output
        r["task_id"] = done.id
        results.append(r)
        print(f"  Task {done.id}: {r}  worker={done.addrport}")
    else:
        print(f"  Task {done.id} failed: {done.result}")

total_words = sum(r.get("word_count", 0) for r in results)
print(f"\nAll {len(results)} task(s) completed — {total_words:,} words processed")